In [ ]:
"""
run_pm.py - 모델링 실행

로컬의 input 데이터로 모델링을 수행하고 output을 생성합니다.
(현재는 목업으로 구현)

Usage:
    from run_pm import run_model
    
    result = run_model(user_id="sean", run_id="20260303T...")
"""

In [ ]:
import os
from datetime import datetime
from run_pm_utils import (
    InputConfig,
    OutputConfig,
    LocalConfig,
    create_file,
    print_header,
    print_separator
)


def get_output_mockups(run_id: str, user_id: str = None) -> dict:
    """Output 결과물 목업 데이터 정의"""
    return {
        'metadata': {
            'run_manifest.yml': {
                'type': 'yaml',
                'content': {
                    'version': '1.0',
                    'run_id': run_id,
                    'created_at': datetime.utcnow().isoformat(),
                    'created_by': user_id or OutputConfig.USER_ID,
                    'status': 'completed',
                    'project': OutputConfig.PROJECT,
                    'experiment': OutputConfig.EXPERIMENT,
                    'model': OutputConfig.MODEL,
                    'algo': OutputConfig.ALGO,
                    'execution': {
                        'instance_type': 'ml.m5.xlarge',
                        'instance_count': 1,
                        'duration_seconds': 1234,
                    }
                }
            }
        },
        'config': {
            'config.yml': {
                'type': 'yaml',
                'content': {
                    'version': '1.0',
                    'algorithm': {'name': 'lightgbm', 'task': 'regression'},
                    'hyperparameters': {
                        'objective': 'regression',
                        'metric': 'rmse',
                        'num_leaves': 31,
                        'learning_rate': 0.05,
                        'n_estimators': 1000,
                        'early_stopping_rounds': 50,
                    }
                }
            }
        },
        'data_refs': {
            'input_data_ref.yml': {
                'type': 'yaml',
                'content': {
                    'version': '1.0',
                    'input_s3_uri': InputConfig.get_s3_uri(user_id=user_id),
                    'data_sources': {
                        'train': {'format': 'parquet', 'rows': 100000},
                        'validation': {'format': 'parquet', 'rows': 10000},
                        'test': {'format': 'parquet', 'rows': 10000},
                    }
                }
            }
        },
        'artifacts/model': {
            '.placeholder': {'type': 'touch', 'content': None}
        },
        'artifacts/metrics': {
            'model_metrics.yml': {
                'type': 'yaml',
                'content': {
                    'version': '1.0',
                    'metrics': {
                        'train': {'rmse': 0.1234, 'mae': 0.0987, 'r2': 0.9123},
                        'validation': {'rmse': 0.1456, 'mae': 0.1123, 'r2': 0.8901},
                        'test': {'rmse': 0.1523, 'mae': 0.1198, 'r2': 0.8856},
                    },
                    'feature_importance': {
                        'day_of_week': 0.25,
                        'is_holiday': 0.20,
                        'temperature': 0.18,
                        'promotion_flag': 0.15,
                        'precipitation': 0.12,
                        'store_id': 0.10,
                    }
                }
            }
        },
        'artifacts/charts': {
            '.placeholder': {'type': 'touch', 'content': None}
        },
        'artifacts/explainability': {
            '.placeholder': {'type': 'touch', 'content': None}
        },
        'reports': {
            'report_request.yml': {
                'type': 'yaml',
                'content': {
                    'version': '1.0',
                    'report_type': 'model_training_summary',
                    'requested_at': datetime.utcnow().isoformat(),
                    'requested_by': user_id or OutputConfig.USER_ID,
                    'include_sections': [
                        'executive_summary',
                        'data_overview',
                        'model_performance',
                        'feature_importance',
                        'recommendations'
                    ]
                }
            }
        }
    }


def create_output_mockups(output_dir: str, mockups: dict) -> list:
    """Output 목업 파일들 생성"""
    created_files = []
    
    for subdir, files in mockups.items():
        subdir_path = os.path.join(output_dir, subdir)
        os.makedirs(subdir_path, exist_ok=True)
        
        for filename, file_info in files.items():
            filepath = os.path.join(subdir_path, filename)
            content = file_info.get('content')
            file_type = file_info.get('type', 'yaml')
            create_file(filepath, content, file_type)
            created_files.append(filepath)
            print(f"  ✓ Created: {filepath}")
    
    return created_files


def run_model(
    user_id: str = None,
    run_id: str = None,
    local_root: str = None
) -> dict:
    """
    모델링 수행 (목업) 및 output 파일 생성
    
    Args:
        user_id: 사용자 ID
        run_id: 실행 ID (없으면 자동 생성)
        local_root: 로컬 루트 디렉토리
    
    Returns:
        dict: {status, run_id, output_dir, created_files}
    """
    local_root = local_root or LocalConfig.ROOT
    local_output_dir = os.path.join(local_root, LocalConfig.OUTPUT_DIR)
    
    run_id = run_id or OutputConfig.generate_run_id()
    
    print_header("Step 2: 모델링 수행")
    print(f"Run ID: {run_id}")
    print(f"Output Dir: {local_output_dir}")
    print_separator()
    
    # 모델링 시뮬레이션
    print("  ... 데이터 로딩 중 ...")
    print("  ... 전처리 중 ...")
    print("  ... 모델 학습 중 (스킵) ...")
    print("  ... 검증 중 (스킵) ...")
    print("  ... 테스트 중 (스킵) ...")
    print("✓ 모델링 완료 (목업)")
    
    # Output 파일 생성
    print("\n[Output 파일 생성]")
    os.makedirs(local_output_dir, exist_ok=True)
    mockups = get_output_mockups(run_id, user_id)
    created_files = create_output_mockups(local_output_dir, mockups)
    
    print(f"\n✓ Created {len(created_files)} output files")
    
    return {
        'status': 'completed',
        'run_id': run_id,
        'output_dir': local_output_dir,
        'created_files': created_files
    }


# ============================================================
# CLI Entry Point
# ============================================================

if __name__ == "__main__":
    import argparse
    
    parser = argparse.ArgumentParser(description="Run model training (mockup)")
    parser.add_argument("--user-id", default="sean", help="User ID")
    parser.add_argument("--run-id", default=None, help="Run ID (auto-generated if not provided)")
    parser.add_argument("--local-root", default=None, help="Local root directory")
    
    args = parser.parse_args()
    
    result = run_model(
        user_id=args.user_id,
        run_id=args.run_id,
        local_root=args.local_root
    )
    
    print(f"\nResult: {result['status']} (run_id: {result['run_id']})")